In [1]:
%load_ext autoreload
%autoreload 2

In [10]:
import sys
import os
import torch
import torchvision
import copy

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.append(project_root)

In [15]:
from src.trainer.BufferTrainer import BufferTrainer
from src.data_utils import (
    get_mnist_tasks,
    _extract_targets,
    get_context_sets,
    create_holdout_set,
)
from src.utils.general import InContextHead, print_colored, set_seed
from src import models
from src.buffer import MultiTaskBuffer
from src.data_utils import split_mnist_by_labels, get_context_sets
from src.regulariser import UnbiasRegulariser, L2Regulariser, MultiRegulariser

from configs import MNIST_BUFFER_CONFIG as CONFIG

In [ ]:
device = "cuda"
classes = [
    "airplane",
    "automobile",
    "bird",
    "cat",
    "deer",
    "dog",
    "frog",
    "horse",
    "ship",
    "truck",
]

task_pairs = [
    ("cat", "truck"),
    ("frog", "ship"),
    ("horse", "automobile"),
    ("dog", "airplane"),
    ("bird", "deer"),
]
task_pairs_ids = [(classes.index(i), classes.index(j)) for i, j in task_pairs]
animals_mask = torch.tensor([0, 0, 1, 1, 1, 1, 1, 1, 0, 0]).to(device)

transform = torchvision.transforms.Compose(
    [
        torchvision.transforms.Resize((224, 224)),
        torchvision.transforms.ToTensor(),
        torchvision.transforms.Normalize(mean=[0.5] * 3, std=[0.5] * 3),
    ]
)


def domain_map_fn(y):
    return animals_mask[y]


@torch.no_grad()
def encode(x, model, device="cuda"):
    # Handle batching to avoid out-of-memory issues
    batch_size = 2048
    features = []
    for i in range(0, len(x), batch_size):
        batch = x[i : i + batch_size].to(device)
        batch_features = model(batch).flatten(start_dim=1).cpu()
        features.append(batch_features)
    return torch.cat(features, dim=0)


def encode_dataset(train_dataset, test_dataset, encoder, device="cuda"):
    train_imgs, train_labels = train_dataset.data, train_dataset.targets
    test_imgs, test_labels = test_dataset.data, test_dataset.targets
    # apply the appropriate scaling and transposition
    train_imgs = (
        torch.tensor(train_imgs, dtype=torch.float32).permute(0, 3, 1, 2) / 255.0
    )
    test_imgs = torch.tensor(test_imgs, dtype=torch.float32).permute(0, 3, 1, 2) / 255.0
    train_imgs = (train_imgs - 0.5) / 0.5
    test_imgs = (test_imgs - 0.5) / 0.5
    train_labels = torch.tensor(train_labels, dtype=torch.int64).squeeze()
    test_labels = torch.tensor(test_labels, dtype=torch.int64).squeeze()

    if encoder is not None:
        train_imgs = encode(train_imgs, encoder, device)
        test_imgs = encode(test_imgs, encoder, device)

    train_dataset = torch.utils.data.TensorDataset(train_imgs, train_labels)
    test_dataset = torch.utils.data.TensorDataset(test_imgs, test_labels)
    return train_dataset, test_dataset


def get_tasks(encoder):
    non_animals = [0, 1, 8, 9]
    animals = [2, 3, 4, 5, 6, 7]

    non_animal_indices = torch.tensor(non_animals)[torch.randperm(4)].tensor_split(5)
    animal_indices = list(torch.tensor(animals)[torch.randperm(6)].tensor_split(5))
    animal_indices.reverse()

    task_pairs_ids = [
        t.tolist() + n.tolist() for t, n in zip(animal_indices, non_animal_indices)
    ]
    print("Contexts:", task_pairs_ids)
    trainset = torchvision.datasets.CIFAR10(
        root="./data", train=True, download=True, transform=transform
    )
    testset = torchvision.datasets.CIFAR10(
        root="./data", train=False, download=True, transform=transform
    )
    trainset.targets = torch.tensor(trainset.targets)
    testset.targets = torch.tensor(testset.targets)

    if encoder is not None:
        trainset, testset = encode_dataset(trainset, testset, encoder)
    test_tasks = [
        split_mnist_by_labels(testset, task_pair_id) for task_pair_id in task_pairs_ids
    ]
    train_tasks = [
        split_mnist_by_labels(trainset, task_pair_id) for task_pair_id in task_pairs_ids
    ]

    return train_tasks, test_tasks

In [8]:
# Get the CIFAR 100 dataset
cifar100_trainset = torchvision.datasets.CIFAR100(
    root="./data", train=True, download=True, transform=transform
)
cifar100_testset = torchvision.datasets.CIFAR100(
    root="./data", train=False, download=True, transform=transform
)

# Convert targets to tensor for consistency
cifar100_trainset.targets = torch.tensor(cifar100_trainset.targets)
cifar100_testset.targets = torch.tensor(cifar100_testset.targets)

# Print dataset info
print(f"CIFAR-100 training set: {len(cifar100_trainset)} images")
print(f"CIFAR-100 test set: {len(cifar100_testset)} images")
print(f"Number of classes: {len(set(cifar100_trainset.targets.tolist()))}")

# Sample a few class names
print(f"Sample classes: {cifar100_trainset.classes[:10]}")

CIFAR-100 training set: 50000 images
CIFAR-100 test set: 10000 images
Number of classes: 100
Sample classes: ['apple', 'aquarium_fish', 'baby', 'bear', 'beaver', 'bed', 'bee', 'beetle', 'bicycle', 'bottle']


In [9]:
# Create a simple function to load the model
save_dir = os.path.join(project_root, "notebooks/saved_models")
model_path = os.path.join(save_dir, "resnet18_cifar100_pretrained.pth")


def load_pretrained_model(model_path, num_classes=100):
    model = torchvision.models.resnet18(weights=None)
    model.fc = torch.nn.Linear(512, num_classes)
    model.load_state_dict(torch.load(model_path))
    return model


model = load_pretrained_model(model_path)

In [11]:
encoder = copy.deepcopy(model).cuda()
encoder.fc = torch.nn.Identity()

In [12]:
X_min, X_max = None, None
for task in get_tasks(encoder):
    X, _ = next(
        iter(torch.utils.data.DataLoader(task[0], batch_size=10000, shuffle=False))
    )
    if X_min is None:
        X_min, X_max = X.min(dim=0).values, X.max(dim=0).values
    else:
        X_min = torch.min(X_min, X.min(dim=0).values)
        X_max = torch.max(X_max, X.max(dim=0).values)
X_min, X_max = X_min.to(device), X_max.to(device)

Contexts: [[7, 9], [4, 8], [2, 1], [6, 0], [3, 5]]


/tmp/ipykernel_2743757/3313364609.py:59: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  train_labels = torch.tensor(train_labels, dtype=torch.int64).squeeze()
/tmp/ipykernel_2743757/3313364609.py:60: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  test_labels = torch.tensor(test_labels, dtype=torch.int64).squeeze()


In [4]:
import wandb

In [ ]:
SMALL = 1000
MEDIUM = 5000
LARGE = 15000


def run_buffer(buffer_size: int, seed: int, config: wandb.config, paradigm="TIL"):
    """
    ---------------------------------------------
    REQUIRED STUFF
    ---------------------------------------------
    """

    device = "cuda"
    classes = [
        "airplane",
        "automobile",
        "bird",
        "cat",
        "deer",
        "dog",
        "frog",
        "horse",
        "ship",
        "truck",
    ]

    task_pairs = [
        ("cat", "truck"),
        ("frog", "ship"),
        ("horse", "automobile"),
        ("dog", "airplane"),
        ("bird", "deer"),
    ]
    task_pairs_ids = [(classes.index(i), classes.index(j)) for i, j in task_pairs]
    animals_mask = torch.tensor([0, 0, 1, 1, 1, 1, 1, 1, 0, 0]).to(device)

    transform = torchvision.transforms.Compose(
        [
            torchvision.transforms.Resize((224, 224)),
            torchvision.transforms.ToTensor(),
            torchvision.transforms.Normalize(mean=[0.5] * 3, std=[0.5] * 3),
        ]
    )

    def domain_map_fn(y):
        return animals_mask[y]

    @torch.no_grad()
    def encode(x, model, device="cuda"):
        # Handle batching to avoid out-of-memory issues
        batch_size = 2048
        features = []
        for i in range(0, len(x), batch_size):
            batch = x[i : i + batch_size].to(device)
            batch_features = model(batch).flatten(start_dim=1).cpu()
            features.append(batch_features)
        return torch.cat(features, dim=0)

    def encode_dataset(train_dataset, test_dataset, encoder, device="cuda"):
        train_imgs, train_labels = train_dataset.data, train_dataset.targets
        test_imgs, test_labels = test_dataset.data, test_dataset.targets
        # apply the appropriate scaling and transposition
        train_imgs = (
            torch.tensor(train_imgs, dtype=torch.float32).permute(0, 3, 1, 2) / 255.0
        )
        test_imgs = (
            torch.tensor(test_imgs, dtype=torch.float32).permute(0, 3, 1, 2) / 255.0
        )
        train_imgs = (train_imgs - 0.5) / 0.5
        test_imgs = (test_imgs - 0.5) / 0.5
        train_labels = torch.tensor(train_labels, dtype=torch.int64).squeeze()
        test_labels = torch.tensor(test_labels, dtype=torch.int64).squeeze()

        if encoder is not None:
            train_imgs = encode(train_imgs, encoder, device)
            test_imgs = encode(test_imgs, encoder, device)

        train_dataset = torch.utils.data.TensorDataset(train_imgs, train_labels)
        test_dataset = torch.utils.data.TensorDataset(test_imgs, test_labels)
        return train_dataset, test_dataset

    def get_tasks(encoder, seed=42):
        set_seed(seed)
        non_animals = [0, 1, 8, 9]
        animals = [2, 3, 4, 5, 6, 7]

        non_animal_indices = torch.tensor(non_animals)[torch.randperm(4)].tensor_split(
            5
        )
        animal_indices = list(torch.tensor(animals)[torch.randperm(6)].tensor_split(5))
        animal_indices.reverse()

        task_pairs_ids = [
            t.tolist() + n.tolist() for t, n in zip(animal_indices, non_animal_indices)
        ]
        print("Contexts:", task_pairs_ids)
        trainset = torchvision.datasets.CIFAR10(
            root="./data", train=True, download=True, transform=transform
        )
        testset = torchvision.datasets.CIFAR10(
            root="./data", train=False, download=True, transform=transform
        )
        trainset.targets = torch.tensor(trainset.targets)
        testset.targets = torch.tensor(testset.targets)

        if encoder is not None:
            trainset, testset = encode_dataset(trainset, testset, encoder)
        test_tasks = [
            split_mnist_by_labels(testset, task_pair_id)
            for task_pair_id in task_pairs_ids
        ]
        train_tasks = [
            split_mnist_by_labels(trainset, task_pair_id)
            for task_pair_id in task_pairs_ids
        ]

        return train_tasks, test_tasks

    # Sample a few class names
    print(f"Sample classes: {cifar100_trainset.classes[:10]}")

    # Create a simple function to load the model
    def load_pretrained_model(model_path, num_classes=100):
        model = torchvision.models.resnet18(weights=None)
        model.fc = torch.nn.Linear(512, num_classes)
        model.load_state_dict(torch.load(model_path))
        return model

    model = load_pretrained_model(model_path)
    encoder = copy.deepcopy(model).cuda()
    encoder.fc = torch.nn.Identity()

    X_min, X_max = None, None
    for task in get_tasks(encoder, seed):
        X, _ = next(
            iter(torch.utils.data.DataLoader(task[0], batch_size=10000, shuffle=False))
        )
        if X_min is None:
            X_min, X_max = X.min(dim=0).values, X.max(dim=0).values
        else:
            X_min = torch.min(X_min, X.min(dim=0).values)
            X_max = torch.max(X_max, X.max(dim=0).values)
    X_min, X_max = X_min.to(device), X_max.to(device)

    """
    ---------------------------------------------
    REQUIRED STUFF
    ---------------------------------------------
    """

    def domain_map_fn(labels: torch.Tensor) -> torch.Tensor:
        """Map the global label to the in context label."""
        return labels % 2

    SEED = seed
    CONFIG = config
    set_seed(SEED)
    train_tasks, test_tasks = get_tasks(encoder, SEED)

    context_sets = get_context_sets(test_tasks)
    head = InContextHead(context_sets, 10, device="cuda")
    head.set_context(0)
    model = models.get_fully_connected_model(
        input_dim=512,
        seed=seed,
        device="cuda",
        output_dim=2 if paradigm == "DIL" else 10,
        head=head if paradigm == "TIL" else None,
    )
    print(
        f"Tasks: {[torch._unique(_extract_targets(train))[0].tolist() for train in train_tasks]}"
    )

    unbias = UnbiasRegulariser(
        lmbd=CONFIG["unbias_lambda"],
        unbias_domain=[X_min, X_max],
    )
    l2 = L2Regulariser(lmbd=CONFIG["l2_lambda"])
    regulariser = MultiRegulariser([l2, unbias])

    if buffer_size == SMALL:
        sizes = [400, 400, 200, 0, 0]
    elif buffer_size == MEDIUM:
        sizes = [1400, 1200, 800, 600, 0]
    elif buffer_size == LARGE:
        sizes = [4800, 4000, 4000, 3200, 0]
    train_tasks, buffer_tasks = zip(
        *[
            create_holdout_set(dataset, holdout_size=holdout)
            for dataset, holdout in zip(train_tasks, sizes)
        ]
    )
    print([len(task) for task in buffer_tasks])
    print([len(task) for task in train_tasks])

    task_labels = [
        torch._unique(_extract_targets(train))[0].tolist() for train in train_tasks
    ]

    buffer = MultiTaskBuffer([])
    buffer_trainer = BufferTrainer(
        model,
        checkpoint=CONFIG["checkpoint"],
        n_iters=CONFIG["n_iters"],
        min_acc_limit=CONFIG["initial_target_acc"],
        min_acc_increment=0,
        primal_learning_rate=CONFIG["primal_learning_rate"],
        dual_learning_rate=CONFIG["dual_learning_rate"],
        projection_strategy=CONFIG["projection_strategy"],
        n_certificate_samples=CONFIG["n_certificate_samples"],
        penalty_coefficient=CONFIG["penalty_coefficient"],
        paradigm=paradigm,
        seed=SEED,
        buffer=buffer,
        domain_map_fn=domain_map_fn if paradigm == "DIL" else None,
        task_labels=task_labels,
    )

    if buffer_size == SMALL:
        MAX_BUFFER_CALLS = 1
    if buffer_size == MEDIUM:
        MAX_BUFFER_CALLS = 3
    if buffer_size == LARGE:
        MAX_BUFFER_CALLS = 7

    targets = {
        "TIL": 0.85,
        "DIL": 0.7,
        "CIL": 0.65
    }
    target_acc = targets[paradigm]
    lower_bounds = []
    buffer_calls = []
    accuracy_matrix = []
    for i, (train, test, buffer) in enumerate(
        zip(train_tasks, test_tasks, buffer_tasks)
    ):
        buffer_trainer.train(
            train,
            test,
            batch_size=CONFIG["batch_size"],
            epochs=CONFIG["epochs"],
            lr=CONFIG["lr"],
            weight_decay=CONFIG["weight_decay"],
            regulariser=regulariser,
            context_id=i if paradigm == "TIL" else None,
        )
        results = buffer_trainer.test(
            test_tasks,
            context_list=list(range(len(test_tasks)))
            if paradigm == "TIL"
            else [None] * len(test_tasks),
        )
        accs = [res[1] for res in results]
        if i == 0 and accs[0] < 0.65:
            wandb.finish(1)
            return
        # If results are not satisfactory, then use buffer data to recompute rashomon set and continue training
        j = 0
        buffer_call = 0
        prev_acc = None
        while (
            j < MAX_BUFFER_CALLS
            and results[i][1] < target_acc
            and i > 0
            and not buffer_trainer.buffer.is_empty()
        ):
            buffer_call += 1
            print_colored("Using buffer to recompute LID.", color="amber")

            buffer_trainer.recall_dataset, (buffer_X, buffer_y) = (
                buffer_trainer.buffer.consume_merge()
            )
            print("Recall dataset size:", len(buffer_trainer.recall_dataset))
            dataset = torch.utils.data.TensorDataset(buffer_X, buffer_y)
            buffer_trainer.compute_rashomon_set(
                dataset,
                use_outer_bbox=False,
                batch_size=len(dataset),
                context_id=i if paradigm == "TIL" else None,
            )
            buffer_trainer.train(
                train,
                test,
                batch_size=CONFIG["batch_size"],
                epochs=CONFIG["epochs"],
                lr=CONFIG["lr"],
                weight_decay=CONFIG["weight_decay"],
                regulariser=regulariser,
                early_stopping=True,
                val_freq=25,
                patience=5,
                context_id=i if paradigm == "TIL" else None,
            )
            results = buffer_trainer.test(
                test_tasks,
                context_list=list(range(len(test_tasks)))
                if paradigm == "TIL"
                else [None] * len(test_tasks),
            )
            accs = [res[1] for res in results]

            print("lower_bounds:", lower_bounds)
            diffs = [accs[i] - lower_bounds[i] for i in range(len(lower_bounds))]
            min_diff_idx = diffs.index(
                min(diffs)
            )  # The assumption is that the task closest to its boundary is the one restricting further improvements
            if results[i][1] > target_acc:
                break
            elif (
                prev_acc is not None
                and results[i][1] - prev_acc < CONFIG["loosening_thresh"]
            ):
                print("Loosening task", min_diff_idx, "bounds.")
                lower_bounds[min_diff_idx] = max(
                    lower_bounds[min_diff_idx] - CONFIG["loosening_step"], 0.01
                )
                buffer_trainer.min_acc_limit = lower_bounds
            prev_acc = results[i][1]
            j += 1
        buffer_calls.append(buffer_call)

        print_colored(accs, color="green")
        accuracy_matrix.append(accs)

        lower_bounds.append(max(accs[i] - CONFIG["min_acc_increment"], 0.01))

        buffer_trainer.min_acc_limit = lower_bounds

        if buffer_trainer._last_projection is not None:
            buffer_trainer.final_certificates.append(
                buffer_trainer.certificates[buffer_trainer._last_projection]
            )
        if i < len(train_tasks) - 1:
            buffer_trainer.compute_rashomon_set(
                test, context_id=i if paradigm == "TIL" else None
            )
            if len(buffer):
                buffer_trainer.add_to_buffer(buffer, task_id=i, k=CONFIG["buffer_k"])
        else:
            print("Buffer calls:", buffer_calls)
            accuracy_matrix.append(buffer_trainer.final_certificates + [0])
            columns = [f"Test Task {i}" for i in range(len(test_tasks))]
            rows = [f"Task {i}" for i in range(len(test_tasks))] + ["Certificates"]
            wandb.log(
                {
                    "accuracy_matrix": wandb.Table(
                        data=accuracy_matrix, columns=columns, rows=rows
                    )
                }
            )

    wandb.finish(0)


for paradigm in ["TIL", "DIL"]:
    for buffer_label, buffer_size in [
        ("small", SMALL),
        ("medium", MEDIUM),
        ("large", LARGE),
    ]:
        start = 0
        if buffer_label == "small":
            start = 1
        for i in range(start, 5):
            with wandb.init(
                project="certified-continual-learning",
                config=CONFIG,
                reinit=True,
                tags=[
                    "final_cifar_buffer",
                    f"buffer_{buffer_label}",
                    f"buffer_{paradigm.lower()}",
                ],
            ):
                wandb.log({"seed": i})
                config = wandb.config
                run_buffer(buffer_size, i, config, paradigm=paradigm)

Sample classes: ['apple', 'aquarium_fish', 'baby', 'bear', 'beaver', 'bed', 'bee', 'beetle', 'bicycle', 'bottle']
Contexts: [[3, 1], [2, 9], [7, 8], [5, 0], [4, 6]]


/tmp/ipykernel_2743757/3055824201.py:71: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  train_labels = torch.tensor(train_labels, dtype=torch.int64).squeeze()
/tmp/ipykernel_2743757/3055824201.py:72: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  test_labels = torch.tensor(test_labels, dtype=torch.int64).squeeze()


Contexts: [[3, 1], [2, 9], [7, 8], [5, 0], [4, 6]]
Tasks: [[1, 3], [2, 9], [7, 8], [0, 5], [4, 6]]
[400, 400, 200, 0, 0]
[9600, 9600, 9800, 10000, 10000]


Training Epochs: 100%|█████████████████████████████████████████████| 5/5 [00:22<00:00,  4.41s/it, val_loss=0.3596, val_acc=0.8475, proj=None, progress=0.99]


Test Results: [(0.3666, 0.841), (2.3026, 0.5), (2.3026, 0.0), (2.3026, 0.5), (2.3026, 0.5)] (Avg: (1.9154, 0.4682))
[0.841, 0.5, 0.0, 0.5, 0.5]
Initial acc constraint violation: -0.1607 (Positive = violated)
Number of model parameters: 5130
Computing Rashomon set with min acc limit: 0.69
Initial bbox:  Obj=16.00,  Size=82080.00,  Min acc hard=0.86,  Min acc soft=0.85


...

Training Epochs: 100%|████████████████████████████████████████████████| 5/5 [00:17<00:00,  3.58s/it, val_loss=0.9731, val_acc=0.8495, proj=0, progress=0.99]


Test Results: [(2.3026, 0.4415), (2.3026, 0.186), (0.4474, 0.84), (0.9855, 0.8555), (2.3026, 0.38)] (Avg: (1.6681, 0.5406))
[0.4415, 0.186, 0.84, 0.8555, 0.38]
Initial acc constraint violation: -0.1351 (Positive = violated)
Computing Rashomon set within outer box of size: 82081.16
Number of model parameters: 5130
Computing Rashomon set with min acc limit: 0.71
Initial bbox:  Obj=16.00,  Size=82080.00,  Min acc hard=0.85,  Min acc soft=0.84


100%|██████████████████████████████████████████████████████████████████████| 200/200 [00:09<00:00, 20.36it/s, size=61562.38, obj=12.000, min_soft_acc=0.763]


Final bbox:  Obj=12.00,  Size=61562.38,  Min acc hard=0.76,  Min acc soft=0.76
Computing final certificates over 400 samples
Checkpointed every 20 iterations for a total of 10 checkpoints
Checkpoints sizes: ['61562.53', '61561.93', '61561.76', '61562.46', '61562.28', '61562.38', '61561.77', '61562.36', '61562.65', '61562.38']
Checkpoint certificates: ['0.69', '0.80', '0.81', '0.75', '0.75', '0.75', '0.81', '0.75', '0.74', '0.76']
----------------------- Finished Computing Rashomon set ------------------------


Training Epochs:  80%|██████████████████████████████████████▍         | 4/5 [00:20<00:04,  4.84s/it, val_loss=1.1403, val_acc=0.7420, proj=0, progress=0.36]